# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [ ]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/
https://edwarddonner.com/2025/11/11/

In [8]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [10]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'linkedin profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
select_relevant_links("https://huggingface.co")

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [11]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [12]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
tencent/HY-MT1.5-1.8B
Updated
6 days ago
•
6.61k
•
641
Qwen/Qwen-Image-2512
Updated
7 days ago
•
16.8k
•
504
LGAI-EXAONE/K-EXAONE-236B-A23B
Updated
1 day ago
•
2.67k
•
417
Lightricks/LTX-2
Updated
about 2 hours ago
•
84.4k
•
373
IQuestLab/IQuest-Coder-V1-40B-Loop-Instruct
Updated
about 2 hours ago
•
6.32k
•
267
Browse 2M+ models
Spaces
Running
Featured
3.85k
Wan2.2 Animate
👁
3.85k
Wan2.2 Animate
Running
on
Zero
1.09k
Z Image Turbo
🖼
1.09k
Generate images from text prompts
Running
on
Zero
MCP
Featured
241
Qwen-Image-Edit-2511-LoRAs-Fast
🎃
241
Demo of the Collection of Qwen Image Edit LoRAs
Running
on
CPU Upgrade
470
Omni Image Edi

In [13]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [14]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [15]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ntencent/HY-MT1.5-1.8B\nUpdated\n6 days ago\n•\n6.61k\n•\n641\nQwen/Qwen-Image-2512\nUpdated\n7 days ago\n•\n16.8k\n•\n504\nLGAI-EXAONE/K-EXAONE-236B-A23B\nUpdated\n1 day ago\n•\n2.67k\n•\n417\nLightricks/LTX-2\nUpdated\nabout 2 hours ago\n•\n84.4k\n•\n374\nIQuestLab/IQuest-Coder-V1-40B-Loop-Instruct\nUpdated\nabout 2 hours ago\n•\n6.32k\n•\n267\nBrowse 2M+ models\nSpaces\nRunning\nFeatured\

In [16]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [17]:
create_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face – Building the Future of AI Together

---

## About Hugging Face

Hugging Face is a vibrant AI community and collaboration platform dedicated to the machine learning (ML) ecosystem. It empowers ML engineers, scientists, and developers worldwide to create, share, and innovate collectively. Hugging Face serves as a central hub—**The Home of Machine Learning**—where the community collaborates on models, datasets, and applications across modalities including text, image, video, audio, and 3D.

By fostering openness and ethical AI development, Hugging Face accelerates AI advancements and invites users to build a transparent and inclusive future for machine learning.

---

## What We Offer

### Extensive Model & Dataset Library  
- Access over **2 million ML models** and **500,000+ datasets** covering a wide range of domains and use cases.  
- Continuously updated collections including popular implementations such as Tencent's HY-MT1.5, Qwen's Qwen-Image-2512, and more.

### Spaces – Showcase and Run AI Applications  
- Host and discover interactive AI demos and applications built by the community.  
- Explore diverse apps like image generation from text, photo editing, AI animation, and other cutting-edge tools.

### Collaboration Platform  
- Host unlimited public machine learning models, datasets, and applications.  
- Build your portfolio by sharing your work and connecting with peers and potential collaborators around the globe.

### Open Source and Enterprise Solutions  
- Leverage the Hugging Face open-source stack to speed up development.  
- Unlock premium paid compute resources and enterprise-grade solutions tailored for teams and organizations.

---

## Community and Culture

Hugging Face is more than a platform—it's a **community-driven ecosystem** focused on openness, collaboration, and ethical AI practice. The company nurtures a culture of sharing knowledge, collective improvement, and innovative exploration.

By empowering the next generation of AI talent, Hugging Face fosters an environment where growth, diversity, and impact are encouraged. Members are invited to contribute, learn, and challenge boundaries in AI development.

---

## Customers and Users

Hugging Face supports a broad spectrum of users including:

- Individual ML engineers and AI researchers  
- Academic institutions and scientists  
- Innovative startups and large enterprises  
- AI ethicists and open-source contributors  
- Developers building AI-powered apps in various modalities  

The platform's flexibility serves learners, practitioners, and enterprises alike, making cutting-edge AI accessible and scalable.

---

## Careers and Opportunities

Joining Hugging Face means becoming part of a passionate mission to build the future of AI collaboratively. The company values diversity, continuous learning, and bold ideas.

Careers at Hugging Face offer opportunities to work on open-source AI projects, collaborate with top AI talent worldwide, and contribute to tools used by millions globally.

Whether your expertise lies in engineering, research, product, or community engagement, Hugging Face welcomes skilled individuals ready to innovate in AI.

---

## Get Involved

- Explore and experiment with thousands of open-source ML models and datasets  
- Create and deploy your own AI applications in Hugging Face Spaces  
- Join a global community of AI professionals and enthusiasts  
- Accelerate your projects with paid Compute and Enterprise solutions  

**Sign Up today** at [huggingface.co](https://huggingface.co) and start building the future of machine learning!

---

## Brand Identity

Hugging Face’s vibrant brand reflects its innovative and approachable nature:

- **Colors:** Yellow (#FFD21E), Orange (#FF9D00), and Gray (#6B7280)  
- Logos available in `.svg`, `.png`, and `.ai` formats for various applications  

---

Hugging Face is shaping an open, ethical, and collaborative AI ecosystem for everyone. Join us in transforming machine learning into a community-powered force for good.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [18]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [19]:
stream_brochure("HuggingFace", "https://huggingface.co")

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the AI community building the future of machine learning. It is a leading collaboration platform where the global machine learning community comes together to create, discover, share, and collaborate on models, datasets, and applications. Leveraging an open-source stack, Hugging Face accelerates AI innovation across all modalities including text, image, video, audio, and even 3D.

---

## Our Platform

- **Models:** Access over 2 million machine learning models updated regularly by contributors worldwide.
- **Datasets:** Browse from over 500,000 datasets spanning a wide range of domains and applications.
- **Spaces:** Host and try thousands of AI-powered applications available to the community.
- **Community:** A growing ecosystem of machine learning engineers, scientists, and enthusiasts collaborating to advance AI.
- **Enterprise & Compute:** Offers tailored paid enterprise solutions enabling teams and organizations to build with state-of-the-art AI tools and infrastructure.

---

## Why Hugging Face?

- **Collaboration at Scale:** Unlimited hosting and sharing of public models and datasets foster a vibrant open-source environment.
- **Multi-Modal AI:** Support for various data types encourages innovation in diverse AI fields.
- **Portfolio Building:** Users can showcase their projects and contributions, advancing their careers in ML.
- **Ethical AI Focus:** Commitment to building an open and ethical AI future.
- **Cutting-edge Technologies:** Stay ahead with access to leading open-source machine learning libraries and tools.

---

## Company Culture

Hugging Face fosters a community-first mindset valuing openness, collaboration, and innovation. It empowers machine learning professionals to learn, experiment, and share freely. The company’s ethos centers around ethical AI development and democratizing access to transformative AI technologies globally.

---

## Customers and Users

Hugging Face serves a diverse set of users including:

- Individual machine learning engineers and researchers
- Startups and AI developers building innovative applications
- Enterprises seeking scalable AI compute and tools for teams
- Educational institutions and students gaining hands-on AI experience
- AI communities and organizations committed to open science and ethical AI

Top trending models and datasets regularly highlight Hugging Face’s ecosystem vitality, featuring contributions from major AI entities like Tencent, Facebook, Wikimedia, and Anthropic.

---

## Careers and Opportunities

Join Hugging Face and become part of the AI revolution! The company offers:

- Roles for software engineers, machine learning researchers, data scientists, and community managers
- Opportunities to work with cutting-edge AI technologies and open-source projects
- A mission-driven workplace promoting collaboration and ethical AI
- Competitive compensation and growth potential in a fast-growing industry

Explore open positions on the Hugging Face website and help build the future of AI community and technology.

---

## Get Involved

- **Sign Up:** Create your free account to start exploring models, datasets, and applications.
- **Contribute:** Share your own projects and collaborate with thousands of AI peers.
- **Enterprise Solutions:** Accelerate your organization’s AI capabilities with dedicated compute and team tools.
- **Explore:** Dive into millions of open-source models and datasets covering every AI niche.

---

## Contact and More Information

Website: [huggingface.co](https://huggingface.co)  
Join the AI community building the future.

---

*Hugging Face — The Home of Machine Learning Collaboration.*

In [21]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("Murphy", "https://getmurphy.ai/")

# Murphy: Boost Your Debt Recovery with AI

## About Murphy
Murphy is an innovative company dedicated to transforming the debt recovery industry through the power of artificial intelligence. By leveraging cutting-edge AI technology, Murphy empowers businesses to enhance their debt recovery processes, making them faster, more efficient, and more successful.

## What We Do
Murphy specializes in intelligent debt recovery solutions that automate and optimize the collection process. Our AI-driven platform helps organizations identify the best strategies for recovering outstanding payments, reducing operational costs, and improving customer engagement during recovery efforts.

## Company Culture
At Murphy, we foster a culture of innovation, collaboration, and continuous learning. Our team is passionate about using technology to solve real-world problems and is committed to delivering exceptional value to our customers. We encourage creativity and support career growth in a dynamic and inclusive environment.

## Our Customers
Murphy serves a diverse range of clients, including financial institutions, service providers, and businesses of all sizes that manage receivables. Our AI solutions are tailored to meet the unique needs of each client, helping them recover debts more effectively while maintaining positive customer relationships.

## Careers at Murphy
We are always looking for talented, forward-thinking individuals to join our team. If you are passionate about AI, technology, and making a meaningful impact in the financial sector, Murphy offers exciting career opportunities with room for professional development. Join us and be part of a company that is redefining debt recovery through innovation.

---

**Contact Murphy today to learn how our AI-powered solutions can boost your debt recovery performance.**

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>